# 1. Loading Dataset

In [1]:
import kagglehub
path = kagglehub.dataset_download("patelris/pokemon-dataset-with-stats-and-types")

100%|██████████| 137k/137k [00:00<00:00, 473kB/s]

Extracting files...


In [2]:
import os
os.listdir(path)

['pokemon_types.csv', 'pokemon_complete.csv']

In [3]:
import pandas as pd
pokemon_file = os.path.join(
    path,
    "pokemon_complete.csv"
)

types_file = os.path.join(
    path,
    "pokemon_types.csv"
)

df_pokemon = pd.read_csv(pokemon_file)
df_types = pd.read_csv(types_file)

print(df_pokemon.head())
print(df_types.head())

print("\nPokemon Columns:")
print(df_pokemon.columns.tolist())

   pokedex_number        name type_1  type_2  hp  attack  defense  sp_attack  \
0               1   Bulbasaur  Grass  Poison  45      49       49         65   
1               2     Ivysaur  Grass  Poison  60      62       63         80   
2               3    Venusaur  Grass  Poison  80      82       83        100   
3               4  Charmander   Fire     NaN  39      52       43         60   
4               5  Charmeleon   Fire     NaN  58      64       58         80   

   sp_defense  speed  ...      shape      egg_groups    habitat  growth_rate  \
0          65     45  ...  quadruped   monster|plant  grassland  medium-slow   
1          80     60  ...  quadruped   monster|plant  grassland  medium-slow   
2         100     80  ...  quadruped   monster|plant  grassland  medium-slow   
3          50     65  ...    upright  monster|dragon   mountain  medium-slow   
4          65     80  ...    upright  monster|dragon   mountain  medium-slow   

  capture_rate base_happiness         

# 2. Cleaning Dataset

In [4]:
df_pokemon.columns = (
    df_pokemon.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

df_types.columns = (
    df_types.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

df_pokemon["name"] = (
    df_pokemon["name"]
    .str.lower()
    .str.strip()
    .str.replace(" ", "-", regex=False)
)

for col in ["type_1", "type_2"]:

    df_pokemon[col] = (
        df_pokemon[col]
        .fillna("none")
        .str.lower()
        .str.strip()
        .str.replace(" ", "-", regex=False)
    )

df_pokemon["abilities"] = (
    df_pokemon["abilities"]
    .fillna("")
    .str.lower()
    .str.strip()
)

df_pokemon["hidden_ability"] = (
    df_pokemon["hidden_ability"]
    .fillna("")
    .str.lower()
    .str.strip()
)

def split_abilities(ability_string):

    if not ability_string:
        return []

    return [
        a.strip().replace(" ", "-")
        for a in ability_string.split("|")
    ]

df_pokemon["ability_list"] = (
    df_pokemon["abilities"]
    .apply(split_abilities)
)

df_pokemon["hidden_ability_list"] = (
    df_pokemon["hidden_ability"]
    .apply(split_abilities)
)

df_pokemon["all_abilities"] = (
    df_pokemon["ability_list"] +
    df_pokemon["hidden_ability_list"]
)

print(df_pokemon[
    [
        "name",
        "abilities",
        "hidden_ability",
        "all_abilities"
    ]
].head())

         name             abilities hidden_ability  \
0   bulbasaur  overgrow|chlorophyll    chlorophyll   
1     ivysaur  overgrow|chlorophyll    chlorophyll   
2    venusaur  overgrow|chlorophyll    chlorophyll   
3  charmander     blaze|solar-power    solar-power   
4  charmeleon     blaze|solar-power    solar-power   

                          all_abilities  
0  [overgrow, chlorophyll, chlorophyll]  
1  [overgrow, chlorophyll, chlorophyll]  
2  [overgrow, chlorophyll, chlorophyll]  
3     [blaze, solar-power, solar-power]  
4     [blaze, solar-power, solar-power]  


# 3. Team Features

In [5]:
STRATEGIES = {

    "rain": {

        "weather_abilities": [
            "drizzle"
        ],

        "weather_abuser_abilities": [
            "swift-swim"
        ]
    },

    "sun": {

        "weather_abilities": [
            "drought"
        ],

        "weather_abuser_abilities": [
            "chlorophyll",
            "protosynthesis"
        ]
    },

    "sand": {

        "weather_abilities": [
            "sand-stream"
        ],

        "weather_abuser_abilities": [
            "sand-rush",
            "sand-force"
        ]
    },

    "snow": {

        "weather_abilities": [
            "snow-warning"
        ],

        "weather_abuser_abilities": [
            "slush-rush"
        ]
    },

    "trick_room": {

        "setters": [
            "hatterene",
            "cresselia",
            "porygon2",
            "bronzong",
            "mimikyu",
            "indeedee-f",
            "farigiraf"
        ],

        "abusers": [
            "ursaluna",
            "amoonguss",
            "iron-hands",
            "hariyama",
            "slowking",
            "clodsire"
        ]
    }
}

In [6]:
import random
def get_pokemon_with_ability(df, abilities):

    abilities = set(abilities)

    result = df[
        df["all_abilities"].apply(
            lambda x: any(a in abilities for a in x)
        )
    ]

    return list(result["name"])


def random_pokemon(df, exclude=[]):

    pool = list(
        set(df["name"]) - set(exclude)
    )

    return random.choice(pool)


def build_team(core_pokemon, df, size=6):

    team = core_pokemon.copy()

    while len(team) < size:

        mon = random_pokemon(
            df,
            exclude=team
        )

        team.append(mon)

    return team

def generate_rain_team(df):

    setters = get_pokemon_with_ability(
        df,
        STRATEGIES["rain"]["weather_abilities"]
    )

    abusers = get_pokemon_with_ability(
        df,
        STRATEGIES["rain"]["weather_abuser_abilities"]
    )

    setter = random.choice(setters)

    selected_abusers = random.sample(
        abusers,
        k=min(2, len(abusers))
    )

    core = [setter] + selected_abusers

    while len(core) < 4:

        extra = random.choice(abusers)

        if extra not in core:
            core.append(extra)

    return build_team(core, df)

def generate_sun_team(df):

    setters = get_pokemon_with_ability(
        df,
        STRATEGIES["sun"]["weather_abilities"]
    )

    abusers = get_pokemon_with_ability(
        df,
        STRATEGIES["sun"]["weather_abuser_abilities"]
    )

    setter = random.choice(setters)

    selected_abusers = random.sample(
        abusers,
        k=min(2, len(abusers))
    )

    core = [setter] + selected_abusers

    while len(core) < 4:

        extra = random.choice(abusers)

        if extra not in core:
            core.append(extra)

    return build_team(core, df)

def generate_sand_team(df):

    setters = get_pokemon_with_ability(
        df,
        STRATEGIES["sand"]["weather_abilities"]
    )

    abusers = get_pokemon_with_ability(
        df,
        STRATEGIES["sand"]["weather_abuser_abilities"]
    )

    setter = random.choice(setters)

    selected_abusers = random.sample(
        abusers,
        k=min(2, len(abusers))
    )

    core = [setter] + selected_abusers

    while len(core) < 4:

        extra = random.choice(abusers)

        if extra not in core:
            core.append(extra)

    return build_team(core, df)

def generate_snow_team(df):

    setters = get_pokemon_with_ability(
        df,
        STRATEGIES["snow"]["weather_abilities"]
    )

    abusers = get_pokemon_with_ability(
        df,
        STRATEGIES["snow"]["weather_abuser_abilities"]
    )

    setter = random.choice(setters)

    selected_abusers = random.sample(
        abusers,
        k=min(2, len(abusers))
    )

    core = [setter] + selected_abusers

    while len(core) < 4:

        extra = random.choice(abusers)

        if extra not in core:
            core.append(extra)

    return build_team(core, df)


# 4. Feature Engineering



In [7]:
def extract_team_features(team, df):
    team = [
        p.lower().replace(" ", "-")
        for p in team
    ]

    selected = df[
        df["name"].isin(team)
    ].copy()

    if len(selected) == 0:
        return None

    features = {}

    features["team_size"] = len(team)

    stats = [
        "hp",
        "attack",
        "defense",
        "sp_attack",
        "sp_defense",
        "speed",
        "base_stat_total"
    ]

    for stat in stats:

        features[f"avg_{stat}"] = (
            selected[stat].mean()
        )

    bulk_series = (
        selected["hp"] +
        selected["defense"] +
        selected["sp_defense"]
    )

    offense_series = (
        selected["attack"] +
        selected["sp_attack"]
    )

    features["avg_bulk"] = bulk_series.mean()
    features["avg_offense"] = offense_series.mean()

    features["fast_count"] = int(
        sum(selected["speed"] >= 100)
    )

    features["slow_count"] = int(
        sum(selected["speed"] <= 60)
    )

    features["very_slow_count"] = int(
        sum(selected["speed"] <= 40)
    )

    features["speed_variance"] = (
        selected["speed"].std()
    )

    features["bulky_count"] = int(
        sum(bulk_series >= 300)
    )

    features["very_bulky_count"] = int(
        sum(bulk_series >= 360)
    )

    features["frail_count"] = int(
        sum(bulk_series <= 220)
    )

    features["bulk_variance"] = (
        bulk_series.std()
    )

    physical_attackers = int(
        sum(selected["attack"] > selected["sp_attack"])
    )

    special_attackers = int(
        sum(selected["sp_attack"] > selected["attack"])
    )

    mixed_attackers = int(
        sum(
            abs(
                selected["attack"] -
                selected["sp_attack"]
            ) <= 15
        )
    )

    features["physical_attackers"] = (
        physical_attackers
    )

    features["special_attackers"] = (
        special_attackers
    )

    features["mixed_attackers"] = (
        mixed_attackers
    )

    features["offense_balance"] = abs(
        selected["attack"].mean() -
        selected["sp_attack"].mean()
    )

    features["defense_balance"] = abs(
        selected["defense"].mean() -
        selected["sp_defense"].mean()
    )

    all_abilities = []

    for abilities in selected["all_abilities"]:

        all_abilities.extend(abilities)

    all_abilities = set(all_abilities)

    for weather in [
        "rain",
        "sun",
        "sand",
        "snow"
    ]:

        setter_abilities = set(
            STRATEGIES[weather][
                "weather_abilities"
            ]
        )

        abuser_abilities = set(
            STRATEGIES[weather][
                "weather_abuser_abilities"
            ]
        )

        setter_count = sum(
            any(
                a in setter_abilities
                for a in abilities
            )
            for abilities in selected[
                "all_abilities"
            ]
        )

        abuser_count = sum(
            any(
                a in abuser_abilities
                for a in abilities
            )
            for abilities in selected[
                "all_abilities"
            ]
        )

        features[
            f"{weather}_setter_count"
        ] = setter_count

        features[
            f"{weather}_abuser_count"
        ] = abuser_count

        features[
            f"{weather}_synergy"
        ] = (
            setter_count * 3 +
            abuser_count
        )

    team_set = set(team)

    tr_setters = set(
        STRATEGIES["trick_room"]["setters"]
    )

    tr_abusers = set(
        STRATEGIES["trick_room"]["abusers"]
    )

    tr_setter_count = len(
        team_set & tr_setters
    )

    tr_abuser_count = len(
        team_set & tr_abusers
    )

    features["trick_room_setter_count"] = (
        tr_setter_count
    )

    features["trick_room_abuser_count"] = (
        tr_abuser_count
    )

    features["trick_room_synergy"] = (
        tr_setter_count * 3 +
        tr_abuser_count
    )

    features["stall_score"] = (
        features["very_bulky_count"] * 2 +
        features["bulky_count"] +
        features["slow_count"]
    )

    features["hyper_offense_score"] = (
        features["fast_count"] * 2 +
        features["frail_count"]
    )

    features["balance_score"] = (
        abs(
            features["physical_attackers"] -
            features["special_attackers"]
        )
    )

    return features

# 5. Data Training

In [8]:
import random
import pandas as pd

ALL_WEATHER_SETTERS = set()

for weather in [
    "rain",
    "sun",
    "sand",
    "snow"
]:

    setters = get_pokemon_with_ability(
        df_pokemon,
        STRATEGIES[weather]["weather_abilities"]
    )

    ALL_WEATHER_SETTERS.update(setters)

TR_SETTERS = set(
    STRATEGIES["trick_room"]["setters"]
)

ALL_SETTERS = (
    ALL_WEATHER_SETTERS |
    TR_SETTERS
)

def generate_trick_room_team(df):

    setter = random.choice(
        STRATEGIES["trick_room"]["setters"]
    )

    abusers = random.sample(
        STRATEGIES["trick_room"]["abusers"],
        k=min(
            3,
            len(
                STRATEGIES["trick_room"]["abusers"]
            )
        )
    )

    core = [setter] + abusers

    slow_pool = df[
        df["speed"] <= 60
    ]

    while len(core) < 6:

        mon = random.choice(
            list(slow_pool["name"])
        )

        if mon not in core:
            core.append(mon)

    return core

def generate_hyper_offense_team(df):

    offensive_pool = df[
        (
            df["speed"] >= 100
        ) &
        (
            (
                df["attack"] +
                df["sp_attack"]
            ) >= 230
        )
    ]

    offensive_pool = offensive_pool[
        ~offensive_pool["name"].isin(
            ALL_SETTERS
        )
    ]

    if len(offensive_pool) < 6:
        return None

    return random.sample(
        list(offensive_pool["name"]),
        6
    )

def generate_stall_team(df):

    bulky_pool = df[
        (
            df["hp"] +
            df["defense"] +
            df["sp_defense"]
        ) >= 320
    ]

    bulky_pool = bulky_pool[
        (
            bulky_pool["speed"] <= 85
        )
    ]

    bulky_pool = bulky_pool[
        ~bulky_pool["name"].isin(
            ALL_SETTERS
        )
    ]

    if len(bulky_pool) < 6:
        return None

    core = random.sample(
        list(bulky_pool["name"]),
        6
    )

    return core

def generate_balance_team(df):

    balance_pool = df[
        (
            (
                df["attack"] +
                df["sp_attack"]
            ) >= 170
        ) &
        (
            (
                df["hp"] +
                df["defense"] +
                df["sp_defense"]
            ) >= 240
        ) &
        (
            df["speed"] >= 65
        ) &
        (
            df["speed"] <= 100
        )
    ]

    balance_pool = balance_pool[
        ~balance_pool["name"].isin(
            ALL_SETTERS
        )
    ]

    if len(balance_pool) < 6:
        return None

    return random.sample(
        list(balance_pool["name"]),
        6
    )

TEAM_GENERATORS = {

    "rain": generate_rain_team,

    "sun": generate_sun_team,

    "sand": generate_sand_team,

    "snow": generate_snow_team,

    "trick_room": generate_trick_room_team,

    "hyper_offense": generate_hyper_offense_team,

    "stall": generate_stall_team,

    "balance": generate_balance_team
}

def build_dataset(
    df,
    samples_per_class=500
):

    X = []
    y = []

    for label, generator in (
        TEAM_GENERATORS.items()
    ):

        print(f"Generating {label} teams...")

        generated = 0

        while generated < samples_per_class:

            try:

                team = generator(df)

                if team is None:
                    continue

                features = extract_team_features(
                    team,
                    df
                )

                if features is None:
                    continue

                X.append(features)

                y.append(label)

                generated += 1

            except Exception as e:

                print(
                    f"Error in {label}:",
                    e
                )

    X = pd.DataFrame(X)

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    ).fillna(0)

    return X, y

X, y = build_dataset(
    df_pokemon,
    samples_per_class=500
)

print(X.head())
print(y[:10])

Generating rain teams...
Generating sun teams...
Generating sand teams...
Generating snow teams...
Generating trick_room teams...
Generating hyper_offense teams...
Generating stall teams...
Generating balance teams...
   team_size     avg_hp  avg_attack  avg_defense  avg_sp_attack  \
0          6  66.000000   90.333333    67.500000      73.833333   
1          6  85.000000   86.166667    75.000000      92.500000   
2          6  59.666667   70.000000    59.166667      60.833333   
3          6  54.166667   62.500000    58.333333      83.333333   
4          6  66.500000   64.833333    65.833333      66.166667   

   avg_sp_defense  avg_speed  avg_base_stat_total    avg_bulk  avg_offense  \
0       69.666667  87.500000           454.833333  203.166667   164.166667   
1       70.833333  79.500000           489.000000  230.833333   178.666667   
2       70.833333  90.166667           410.666667  189.666667   130.833333   
3       65.833333  85.666667           409.833333  178.333333   145

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()

y_encoded = le.fit_transform(y)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Train model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=12, min_samples_split=5, n_estimators=300,
                       random_state=42)

# 6. Model Evaluation

In [10]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)

print("ACCURACY:")
print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print("\nCLASSIFICATION REPORT:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=le.classes_
    )
)

ACCURACY:
1.0

CLASSIFICATION REPORT:
               precision    recall  f1-score   support

      balance       1.00      1.00      1.00       100
hyper_offense       1.00      1.00      1.00       100
         rain       1.00      1.00      1.00       100
         sand       1.00      1.00      1.00       100
         snow       1.00      1.00      1.00       100
        stall       1.00      1.00      1.00       100
          sun       1.00      1.00      1.00       100
   trick_room       1.00      1.00      1.00       100

     accuracy                           1.00       800
    macro avg       1.00      1.00      1.00       800
 weighted avg       1.00      1.00      1.00       800



In [11]:
import pandas as pd

cm = confusion_matrix(
    y_test,
    y_pred
)

cm_df = pd.DataFrame(
    cm,
    index=le.classes_,
    columns=le.classes_
)

print(cm_df)

               balance  hyper_offense  rain  sand  snow  stall  sun  \
balance            100              0     0     0     0      0    0   
hyper_offense        0            100     0     0     0      0    0   
rain                 0              0   100     0     0      0    0   
sand                 0              0     0   100     0      0    0   
snow                 0              0     0     0   100      0    0   
stall                0              0     0     0     0    100    0   
sun                  0              0     0     0     0      0  100   
trick_room           0              0     0     0     0      0    0   

               trick_room  
balance                 0  
hyper_offense           0  
rain                    0  
sand                    0  
snow                    0  
stall                   0  
sun                     0  
trick_room            100  


# 7. Validation

In [12]:
def validate_team(team):

    if len(team) != 6:

        return (
            False,
            "Team must contain exactly 6 Pokémon."
        )

    normalized_team = [
        p.lower().replace(" ", "-")
        for p in team
    ]

    if len(set(normalized_team)) != 6:

        return (
            False,
            "Duplicate Pokémon detected."
        )

    valid_names = set(
        df_pokemon["name"]
    )

    invalid = []

    for mon in normalized_team:

        if mon not in valid_names:
            invalid.append(mon)

    if invalid:

        return (
            False,
            f"Invalid Pokémon: {invalid}"
        )

    return (
        True,
        "Valid team."
    )

# 8. Prediction System

In [13]:
def predict_team(team):

    valid, message = validate_team(team)

    if not valid:

        return {
            "success": False,
            "error": message
        }

    features = extract_team_features(
        team,
        df_pokemon
    )

    if features is None:

        return {
            "success": False,
            "error": "Feature extraction failed."
        }

    X_input = pd.DataFrame([features])

    X_input = X_input.reindex(
        columns=X.columns,
        fill_value=0
    )

    prediction_encoded = model.predict(
        X_input
    )[0]

    probabilities = model.predict_proba(
        X_input
    )[0]

    prediction = le.inverse_transform(
        [prediction_encoded]
    )[0]

    probability_dict = {}

    for label, prob in zip(
        le.classes_,
        probabilities
    ):

        probability_dict[label] = round(
            float(prob),
            3
        )

    explanations = explain_prediction(
        team,
        features
    )

    return {

        "success": True,

        "team": team,

        "prediction": prediction,

        "probabilities": probability_dict,

        "explanations": explanations
    }

In [14]:
def explain_prediction(team, features):

    explanations = []

    if features["rain_synergy"] >= 4:

        explanations.append(
            "Strong rain synergy detected."
        )

    if features["sun_synergy"] >= 4:

        explanations.append(
            "Strong sun synergy detected."
        )

    if features["sand_synergy"] >= 4:

        explanations.append(
            "Strong sand synergy detected."
        )

    if features["snow_synergy"] >= 4:

        explanations.append(
            "Strong snow synergy detected."
        )

    if features["trick_room_synergy"] >= 4:

        explanations.append(
            "Strong Trick Room synergy detected."
        )

    if features["fast_count"] >= 4:

        explanations.append(
            "Team contains many fast attackers."
        )

    if features["very_bulky_count"] >= 3:

        explanations.append(
            "Team contains multiple bulky walls."
        )

    if features["hyper_offense_score"] >= 8:

        explanations.append(
            "High offensive pressure detected."
        )

    if features["stall_score"] >= 8:

        explanations.append(
            "High defensive/stall presence detected."
        )

    if len(explanations) == 0:

        explanations.append(
            "Team has mixed archetype characteristics."
        )

    return explanations

# 9. Testing Teams

In [15]:
stall_team = [
    "toxapex",
    "blissey",
    "corviknight",
    "clodsire",
    "dondozo",
    "alomomola"
]

resultstall = predict_team(stall_team)

print("\nPrediction:")
print(resultstall["prediction"])

print("\nProbabilities:")
print(resultstall["probabilities"])

print("\nExplanations:")
print(resultstall["explanations"])

rain_team = [
    "pelipper",
    "barraskewda",
    "kingdra",
    "ferrothorn",
    "zapdos",
    "swampert"
]

resultrain = predict_team(rain_team)

print("\nPrediction:")
print(resultrain["prediction"])

print("\nProbabilities:")
print(resultrain["probabilities"])

print("\nExplanations:")
print(resultrain["explanations"])

ho_team = [
    "dragapult",
    "iron-valiant",
    "roaring-moon",
    "weavile",
    "greninja",
    "garchomp"
]

resultho = predict_team(ho_team)

print("\nPrediction:")
print(resultho["prediction"])

print("\nProbabilities:")
print(resultho["probabilities"])

print("\nExplanations:")
print(resultho["explanations"])

balance_team = [
    "great-tusk",
    "heatran",
    "rotom-wash",
    "dragapult",
    "amoonguss",
    "ting-lu"
]

resultbalance = predict_team(balance_team)

print("\nPrediction:")
print(resultbalance["prediction"])

print("\nProbabilities:")
print(resultbalance["probabilities"])

print("\nExplanations:")
print(resultbalance["explanations"])


Prediction:
stall

Probabilities:
{np.str_('balance'): 0.26, np.str_('hyper_offense'): 0.009, np.str_('rain'): 0.077, np.str_('sand'): 0.068, np.str_('snow'): 0.104, np.str_('stall'): 0.294, np.str_('sun'): 0.092, np.str_('trick_room'): 0.097}

Explanations:
['High defensive/stall presence detected.']

Prediction:
rain

Probabilities:
{np.str_('balance'): 0.105, np.str_('hyper_offense'): 0.007, np.str_('rain'): 0.703, np.str_('sand'): 0.023, np.str_('snow'): 0.041, np.str_('stall'): 0.048, np.str_('sun'): 0.05, np.str_('trick_room'): 0.023}

Explanations:
['Strong rain synergy detected.']

Prediction:
hyper_offense

Probabilities:
{np.str_('balance'): 0.064, np.str_('hyper_offense'): 0.893, np.str_('rain'): 0.003, np.str_('sand'): 0.007, np.str_('snow'): 0.003, np.str_('stall'): 0.0, np.str_('sun'): 0.023, np.str_('trick_room'): 0.007}

Explanations:
['Team contains many fast attackers.', 'High offensive pressure detected.']

Prediction:
balance

Probabilities:
{np.str_('balance'): 0.

# 10. Save Model

In [16]:
import joblib

joblib.dump(model, "pokemon_model.pkl")
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']